# 10 — 3D raw image workflow without a mass spectrum

This notebook is for the fallback case where you have only the per-layer FPD raw image files:

```text
X ; Y ; Channel1 ; Channel2 ; Channel3 ; ...
```

and no calibrated mass-spectrum CSV.

In this case, the scientifically safe workflow is:

```text
raw image layers
→ channel bins
→ 3D channel-filtered ion volumes
```

Element/mass assignment is only possible if you later provide a valid channel→mass calibration.
This notebook therefore supports two modes:

1. **Channel-only mode** — no mass calibration, bins defined by detector channel.
2. **Optional calibration mode** — load or generate a calibration CSV and define bins by mass.

In [ ]:
%load_ext autoreload
%autoreload 2

from pathlib import Path
import re

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from pymagsims import SIMSVolume
from pymagsims.plotting import plot_ion_image_grid, plot_volume_slice

DATA = Path("../data")
RAW_LAYER_DIR = DATA / "3d"

## 1. Natural sorting for raw image layer files

A normal string sort gives:

```text
Image_1.raw, Image_10.raw, Image_11.raw, ..., Image_2.raw
```

We use natural sorting so the layer order is correct.

In [ ]:
def natural_sort_key(path):
    return [
        int(text) if text.isdigit() else text.lower()
        for text in re.split(r"(\d+)", path.name)
    ]

paths = sorted(
    RAW_LAYER_DIR.glob("*Image_*.raw"),
    key=natural_sort_key,
)

print(f"Found {len(paths)} raw image layers")
for p in paths:
    print(p.name)

In [ ]:
from pymagsims.raw_image import SIMSRawImage
import pandas as pd
import matplotlib.pyplot as plt

# Sum channel counts across all raw image layers
all_counts = []

for path in paths:
    raw = SIMSRawImage.from_fpd_raw(path, shape=(256, 256))
    counts = raw.events["Channel"].value_counts()
    all_counts.append(counts)

channel_counts = (
    pd.concat(all_counts, axis=1)
    .fillna(0)
    .sum(axis=1)
    .sort_index()
)

channel_spectrum = pd.DataFrame(
    {
        "Channel": channel_counts.index.astype(int),
        "Counts": channel_counts.values.astype(int),
    }
)

display(channel_spectrum.head())

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))

ax.plot(
    channel_spectrum["Channel"],
    channel_spectrum["Counts"],
    linewidth=1,
)

ax.set_xlabel("Detector channel")
ax.set_ylabel("Total counts")
ax.set_title("Total counts vs detector channel")
ax.set_yscale("log")
ax.grid(True)

fig.tight_layout()

In [ ]:
import plotly.graph_objects as go

fig = go.Figure()

fig.add_trace(
    go.Scatter(
        x=channel_spectrum["Channel"],
        y=channel_spectrum["Counts"],
        mode="lines",
        name="Total counts",
        hovertemplate=(
            "Channel: %{x}<br>"
            "Counts: %{y}<extra></extra>"
        ),
    )
)

fig.update_layout(
    title="Total counts vs detector channel",
    xaxis_title="Detector channel",
    yaxis_title="Total counts",
    template="plotly_white",
    hovermode="closest",
)

fig.update_yaxes(type="log")

fig

## 2. Define channel bins manually

Without mass calibration, define bins directly in detector-channel space.

These labels are intentionally neutral (`channel_bin_*`) unless you know the calibration.

In [ ]:
channel_bins = pd.DataFrame(
    [
        {"label": "channel_bin_1", "ch_min": 1, "ch_max": 500},
        {"label": "channel_bin_2", "ch_min": 501, "ch_max": 1500},
        {"label": "channel_bin_3", "ch_min": 1501, "ch_max": 3000},
        {"label": "channel_bin_4", "ch_min": 8300, "ch_max": 9000},
        {"label": "Au?", "ch_min": 9200, "ch_max": 9500},
    ]
)

display(channel_bins)

###### 3. Build 3D volumes from raw image layers using channel bins

Set the correct image size from the acquisition.

For the example files discussed earlier, the layer size was:

```python
shape=(256, 256)
```

Change this to `(512, 512)` or another size if needed.

In [ ]:
volume = SIMSVolume.from_fpd_raw_image_series(
    paths=paths,
    bins=channel_bins,
    spectrum=None,
    include_total=True,
    shape=(256, 256),
)

print(volume.labels())
display(volume.metadata)

## 4. Confirm 3D arrays

Each label is stored as a separate 3D array:

```text
volume.volumes[label].shape = (z, y, x)
```

In [ ]:
for label, arr in volume.volumes.items():
    print(f"{label}: shape={arr.shape}, total_counts={arr.sum()}")

## 5. Plot slices

In [ ]:
plot_volume_slice(
    volume,
    label="Total",
    z=0,
    log=True,
    cmap="viridis",
);

In [ ]:
label = [l for l in volume.labels() if l != "Total"][0]

plot_volume_slice(
    volume,
    label=label,
    z=0,
    log=True,
    cmap="magma",
);

## 6. Plot summed projections

This collapses the stack along the depth/layer axis.

In [ ]:
projection_images = {
    label: volume.sum_projection(label)
    for label in volume.labels()
}

plot_ion_image_grid(
    projection_images,
    log=True,
    ncols=3,
    cmaps=["gray", "viridis", "magma", "plasma", "cividis", "inferno", "turbo"],
);

## 7. Plot depth profiles from channel bins

These profiles show integrated counts per layer for each channel bin.

In [ ]:
profiles = []

for label in volume.labels():
    profile = volume.depth_profile(label)
    profile = profile.rename(columns={label: "Intensity"})
    profile["Label"] = label
    profiles.append(profile)

depth_profiles = pd.concat(profiles, ignore_index=True)

display(depth_profiles.head())

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))

for label, group in depth_profiles.groupby("Label"):
    ax.plot(
        group["Slice"],
        group["Intensity"],
        marker="o",
        linewidth=1,
        label=label,
    )

ax.set_xlabel("Slice / layer")
ax.set_ylabel("Integrated counts")
ax.set_yscale("log")
ax.grid(True)
ax.legend()

fig.tight_layout()

## 8. Optional: create a placeholder calibration CSV

If no calibration file is available, you can create a **placeholder** channel→mass calibration.

⚠️ This is not a real calibration. It is only useful for testing code paths.

The example below uses an approximate linear mapping from the earlier observed mass range:

```text
channel 1     → mass ≈ 1.700559 amu
channel 12000 → mass ≈ 126.608148 amu
```

Replace this with a real calibration CSV when available.

In [ ]:
calibration_dir = DATA / "calibration"
calibration_dir.mkdir(parents=True, exist_ok=True)

calibration_file = calibration_dir / "placeholder_channel_mass_calibration.csv"

n_channels = 12000
channels = np.arange(1, n_channels + 1)

# Approximate values based on earlier files; not valid for scientific assignment.
mass_min = 10
mass_max = 210

masses = np.linspace(mass_min, mass_max, n_channels)

calibration_df = pd.DataFrame(
    {
        "Channel": channels,
        "Mass": masses,
    }
)

calibration_df.to_csv(calibration_file, index=False)

print(f"Wrote placeholder calibration to: {calibration_file}")
display(calibration_df.head())
display(calibration_df.tail())

## 9. Load a calibration CSV and define mass bins

A calibration CSV should contain at least:

```text
Channel,Mass
```

Once loaded, you can define mass bins and convert them to channel bins.

In [ ]:
calibration_df = pd.read_csv(calibration_file)

display(calibration_df)

In [ ]:
def mass_to_channel_from_calibration(calibration_df, mass):
    idx = (calibration_df["Mass"] - mass).abs().idxmin()
    return int(calibration_df.loc[idx, "Channel"])


def mass_bins_to_channel_bins(mass_bins, calibration_df):
    converted = mass_bins.copy()

    converted["ch_min"] = converted["mass_min"].apply(
        lambda m: mass_to_channel_from_calibration(calibration_df, m)
    )
    converted["ch_max"] = converted["mass_max"].apply(
        lambda m: mass_to_channel_from_calibration(calibration_df, m)
    )

    # Ensure ch_min <= ch_max
    ch_low = converted[["ch_min", "ch_max"]].min(axis=1)
    ch_high = converted[["ch_min", "ch_max"]].max(axis=1)

    converted["ch_min"] = ch_low
    converted["ch_max"] = ch_high

    return converted

## 10. Example mass bins using placeholder calibration

Replace these with real mass bins when a proper calibration is available.

In [ ]:
mass_bins = pd.DataFrame(
    [
        {"label": "mass_bin_around_16", "mass_min": 15.8, "mass_max": 16.2},
        {"label": "mass_bin_around_28", "mass_min": 27.8, "mass_max": 28.2},
        {"label": "mass_bin_around_69", "mass_min": 68.8, "mass_max": 69.2},
        {"label": "mass_bin_around_206", "mass_min": 190, "mass_max": 210},
    ]
)

converted_bins = mass_bins_to_channel_bins(mass_bins, calibration_df)

display(converted_bins)

## 11. Build volume using converted mass bins

This is the same raw-image-only reconstruction, but the bins were defined in mass space
and converted to channels using the calibration table.

With the placeholder calibration, this is only a software test.
With a real calibration, this becomes a calibrated mass-filtered 3D workflow.

In [ ]:
volume_from_calibration = SIMSVolume.from_fpd_raw_image_series(
    paths=paths,
    bins=converted_bins,
    spectrum=None,
    include_total=True,
    shape=(256, 256),
)

print(volume_from_calibration.labels())

In [ ]:
projection_images_calibrated = {
    label: volume_from_calibration.sum_projection(label)
    for label in volume_from_calibration.labels()
}

plot_ion_image_grid(
    projection_images_calibrated,
    log=True,
    ncols=2,
    cmaps=["gray", "viridis", "magma", "plasma"],
);

## 12. Interactive Plotly layer slider

Use this helper to scroll through layers of any label.

In [ ]:
def plot_volume_slider(volume, label="Total", log=True, colorscale="Viridis"):
    import plotly.graph_objects as go
    import numpy as np

    arr = volume.get(label)
    arr_plot = np.log1p(arr) if log else arr

    z_count = arr_plot.shape[0]

    fig = go.Figure()

    for z in range(z_count):
        fig.add_trace(
            go.Heatmap(
                z=arr_plot[z],
                colorscale=colorscale,
                visible=(z == 0),
                colorbar=dict(title="log(1 + counts)" if log else "counts"),
            )
        )

    steps = []
    for z in range(z_count):
        steps.append(
            dict(
                method="update",
                args=[
                    {"visible": [i == z for i in range(z_count)]},
                    {"title": f"{label} — layer {z}"}
                ],
                label=str(z),
            )
        )

    fig.update_layout(
        title=f"{label} — layer 0",
        xaxis_title="X pixel",
        yaxis_title="Y pixel",
        yaxis=dict(scaleanchor="x", autorange="reversed"),
        width=700,
        height=700,
        sliders=[
            dict(
                active=0,
                currentvalue={"prefix": "Layer: "},
                pad={"t": 50},
                steps=steps,
            )
        ],
    )

    return fig

In [ ]:
volume.labels()

In [ ]:
plot_volume_slider(volume, label="channel_bin_1", log=True)

In [ ]:
plot_volume_slider(
    volume_from_calibration,
    label="mass_bin_around_206",
    log=True,
    colorscale="Magma",
)

## Summary

Use this notebook when the calibrated spectrum CSV is missing.

Recommended interpretation:

- Channel-only bins are valid for relative comparison and image reconstruction.
- Element names should not be assigned unless a real calibration is loaded.
- Placeholder calibration is for testing only.
- Replace the placeholder calibration with a real channel→mass CSV as soon as possible.